# Step 1: Data Acquisition

**Paper:** Behavioral Transparency in Multi-Agent RL: Strategic Decoupling and the Economic Optimization of Tactical Shocks  
**Author:** Kenny Ching — School of Business, University of Auckland  

---

## Overview

This notebook handles all raw data acquisition for the study. It has three parts:

| Part | Description | Output |
|---|---|---|
| **A** | Download TI8/TI9 match telemetry from OpenDota API | `data/raw/*.json` |
| **B** | Document and upload OpenAI Five `.dem` replay files | 3 × `.dem` files in `/content/` |
| **C** | Parse `.dem` files with Clarity to produce combat logs | `*_combatlog.txt` files |

Parts A and B produce the inputs for Step 2 (feature extraction).

---

## Part A: Human Baseline — OpenDota API

**Source:** OpenDota public API (https://api.opendota.com)  
**No API key required.** Rate limit: 60 requests/minute for unauthenticated users.  
**Expected runtime:** ~15–20 minutes for TI8 + TI9 Main Event matches.

We restrict to **TI8 (league 9870)** and **TI9 (league 10749)** because these tournaments
represent the apex of human performance during the exact patch era in which OpenAI Five
was trained. Earlier TIs (TI6, TI7) predate the patch; later ones (TI10+) postdate it.

---

## Part B: AI Treatment — OpenAI Five Replay Files

**Source:** OpenAI official blog post  
**URL:** https://openai.com/index/openai-five-defeats-dota-2-world-champions/  

OpenAI released three replay files from the April 2019 Finals event where OpenAI Five
played against and alongside the world champion team OG:

| File | Match | Result |
|---|---|---|
| `og_game_1_04132013569395266.dem` | OG (Radiant) vs OpenAI Five (Dire) | OG wins |
| `og_game2_04132030946085141.dem` | OpenAI Five (Radiant) vs OG (Dire) | OpenAI wins |
| `coop_game_04132076146922985.dem` | OpenAI Five vs OpenAI Five (Coop exhibition) | Radiant wins |

These files must be **manually downloaded** from the OpenAI blog post above and uploaded
to this notebook when prompted in Part B.

---

## Part C: Parsing — Clarity Combat Log Parser

**Source:** https://github.com/skadistats/clarity-examples  
The Clarity Java library parses Source 2 `.dem` files and outputs a structured combat log
containing every hero kill, gold transaction, item purchase, and game state event with
millisecond-precision timestamps. We use the bundled `combatlog` example.

**Requirements:** Java 21 (available by default in Colab)

In [ ]:
# ============================================================
# CELL 1 — Imports and directory setup
# ============================================================
import os, json, time, re, subprocess, zipfile
import requests
from requests.adapters import HTTPAdapter, Retry
from google.colab import files

RAW_DIR     = '/content/data/raw'
COMBATLOG_DIR = '/content/data/combatlog'
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(COMBATLOG_DIR, exist_ok=True)

# Retry session for OpenDota API
session = requests.Session()
retries = Retry(total=5, backoff_factor=1, status_forcelist=[429, 500, 502, 503, 504])
session.mount('https://', HTTPAdapter(max_retries=retries))

print('Directories ready:')
print(f'  Raw JSON:   {RAW_DIR}')
print(f'  Combatlog:  {COMBATLOG_DIR}')

---
## PART A: Download TI8 + TI9 Match Data from OpenDota

In [ ]:
# ============================================================
# CELL 2 — Fetch match IDs for TI8 and TI9
# ============================================================
OPENDOTA_API = 'https://api.opendota.com/api'

# TI8 = league 9870, TI9 = league 10749
# TI6/7 excluded (pre-patch), TI10+ excluded (post-patch)
TI_LEAGUES = {9870: 'TI8', 10749: 'TI9'}

def fetch_league_match_ids(league_id):
    url  = f'{OPENDOTA_API}/leagues/{league_id}/matches'
    resp = session.get(url, timeout=30)
    resp.raise_for_status()
    return [m['match_id'] for m in resp.json()]

all_match_ids = []
for league_id, label in TI_LEAGUES.items():
    try:
        ids = fetch_league_match_ids(league_id)
        # Exclude any OpenAI matches if they appear in the league listing
        ids = [i for i in ids if i not in OPENAI_IDS]
        print(f'{label} (league {league_id}): {len(ids)} matches')
        all_match_ids.extend(ids)
    except Exception as e:
        print(f'{label}: FAILED — {e}')

# Deduplicate
all_match_ids = list(set(all_match_ids))
print(f'\nTotal unique matches to download: {len(all_match_ids)}')

In [ ]:
# ============================================================
# CELL 3 — Download match JSONs from OpenDota
# Expected runtime: ~15-20 minutes
# Resumes automatically if interrupted (skips existing files)
# ============================================================
failed = []
skipped = 0
downloaded = 0

print(f'Downloading {len(all_match_ids)} matches to {RAW_DIR}...')
print('(Will skip files already downloaded — safe to re-run if interrupted)')

for i, mid in enumerate(all_match_ids):
    path = f'{RAW_DIR}/{mid}.json'
    if os.path.exists(path):
        skipped += 1
        continue
    try:
        resp = session.get(f'{OPENDOTA_API}/matches/{mid}', timeout=30)
        if resp.status_code == 200:
            data = resp.json()
            # Only save matches with the gold advantage and teamfight data we need
            if data.get('radiant_gold_adv') and data.get('teamfights'):
                with open(path, 'w') as f:
                    json.dump(data, f)
                downloaded += 1
        time.sleep(0.8)  # Respect OpenDota rate limit (~60 req/min)
    except Exception as e:
        failed.append(mid)

    if (i + 1) % 50 == 0:
        saved = len(os.listdir(RAW_DIR))
        print(f'  {i+1}/{len(all_match_ids)} processed — {saved} files saved')

total_saved = len(os.listdir(RAW_DIR))
print(f'\n✅ Done.')
print(f'  Downloaded this run: {downloaded}')
print(f'  Skipped (existing):  {skipped}')
print(f'  Failed:              {len(failed)}')
print(f'  Total files on disk: {total_saved}')
if failed:
    print(f'  Failed IDs: {failed[:10]}{'...' if len(failed) > 10 else ''}')

---
## PART B: Upload OpenAI Five Replay Files (.dem)

Download the three `.dem` files from the OpenAI blog post before running this cell:

**→ https://openai.com/index/openai-five-defeats-dota-2-world-champions/**

Look for the "Download replays" section on that page. The three files are:
- `og_game_1_04132013569395266.dem` (~41 MB)
- `og_game2_04132030946085141.dem` (~35 MB)
- `coop_game_04132076146922985.dem` (~28 MB)

In [ ]:
# ============================================================
# CELL 4 — Upload .dem files
# ============================================================
print('Upload all 3 .dem replay files from the OpenAI blog post:')
print('https://openai.com/index/openai-five-defeats-dota-2-world-champions/')
print()

uploaded = files.upload()

DEM_PATHS = {}
for fname, data in uploaded.items():
    dest = f'/content/{fname}'
    with open(dest, 'wb') as f:
        f.write(data)
    size_mb = os.path.getsize(dest) / 1024 / 1024

    # Auto-detect which game each file is
    fl = fname.lower()
    if 'game_1' in fl or 'game1' in fl or '13569' in fl:
        label = 'og_game1'
    elif 'game2' in fl or 'game_2' in fl or '30946' in fl:
        label = 'og_game2'
    elif 'coop' in fl or '76146' in fl:
        label = 'coop'
    else:
        label = fname.replace('.dem', '')

    DEM_PATHS[label] = dest
    print(f'  {label}: {fname} ({size_mb:.1f} MB)')

missing = [k for k in ['og_game1','og_game2','coop'] if k not in DEM_PATHS]
if missing:
    print(f'\nWARNING: Could not auto-detect: {missing}')
    print('Manually assign: DEM_PATHS["og_game1"] = "/content/filename.dem" etc.')
else:
    print('\nAll 3 files detected correctly. Proceed to Part C.')

---
## PART C: Parse .dem Files with Clarity

In [ ]:
# ============================================================
# CELL 5 — Verify Java and build Clarity combatlog jar
# Expected build time: ~2 minutes (first run only)
# ============================================================
result = subprocess.run(['java', '-version'], capture_output=True, text=True)
java_ver = result.stderr.strip().split('\n')[0]
print(f'Java: {java_ver}')

CLARITY_DIR = '/content/clarity-examples'
JAR_PATH    = f'{CLARITY_DIR}/build/libs/combatlog.jar'

if not os.path.exists(JAR_PATH):
    print('\nCloning clarity-examples...')
    subprocess.run(
        ['git', 'clone', '--depth=1',
         'https://github.com/skadistats/clarity-examples.git', CLARITY_DIR],
        check=True
    )

    # Patch build file to use Java 21 (Colab default)
    build_file = f'{CLARITY_DIR}/build.gradle.kts'
    with open(build_file) as f:
        content = f.read()
    content = content.replace(
        'languageVersion.set(JavaLanguageVersion.of(17))',
        'languageVersion.set(JavaLanguageVersion.of(21))'
    )
    with open(build_file, 'w') as f:
        f.write(content)
    print('Patched build.gradle.kts for Java 21')

    print('Building combatlog jar (~2 minutes)...')
    result = subprocess.run(
        ['./gradlew', 'combatlogPackage'],
        cwd=CLARITY_DIR, capture_output=True, text=True
    )
    if result.returncode != 0:
        print('BUILD FAILED:')
        print(result.stderr[-3000:])
    else:
        print('Build successful.')
else:
    print(f'JAR already built: {JAR_PATH}')

print(f'\nJAR ready: {os.path.exists(JAR_PATH)}')

In [ ]:
# ============================================================
# CELL 6 — Run Clarity on all 3 .dem files (~1-2 min each)
# ============================================================
print('Parsing replay files...')

for label, dem_path in DEM_PATHS.items():
    out_path = f'{COMBATLOG_DIR}/{label}_combatlog.txt'
    print(f'\n  {label}: {dem_path}')
    result = subprocess.run(
        ['java', '-jar', JAR_PATH, dem_path],
        capture_output=True, text=True, timeout=300
    )
    with open(out_path, 'w') as f:
        f.write(result.stdout)
    lines   = result.stdout.count('\n')
    size_kb = os.path.getsize(out_path) // 1024
    print(f'  -> {out_path} ({lines:,} lines, {size_kb} KB)')

print('\n✅ All combatlog files generated.')

In [ ]:
# ============================================================
# CELL 7 — Verify combatlog output (sanity check)
# ============================================================
for label in ['og_game1', 'og_game2', 'coop']:
    path = f'{COMBATLOG_DIR}/{label}_combatlog.txt'
    if not os.path.exists(path):
        print(f'{label}: FILE MISSING'); continue

    with open(path) as f:
        lines = f.readlines()

    hero_kills  = sum(1 for l in lines
                      if 'is killed by npc_dota_hero' in l
                      and 'npc_dota_hero' in l.split('is killed')[0])
    gold_events = sum(1 for l in lines if 'receives' in l and 'gold' in l)
    state4      = next((l.strip() for l in lines if 'game state is now 4' in l), 'NOT FOUND')
    state6      = next((l.strip() for l in lines if 'game state is now 6' in l), 'NOT FOUND')

    print(f'\n{label}:')
    print(f'  Total lines:  {len(lines):,}')
    print(f'  Hero kills:   {hero_kills}')
    print(f'  Gold events:  {gold_events}')
    print(f'  Game start:   {state4}')
    print(f'  Game end:     {state6}')

# Expected values for reference:
# og_game1: ~77,650 lines, 81 hero kills, 2,538 gold events, ~39.8 min
# og_game2: ~36,282 lines, 46 hero kills, 1,168 gold events, ~22.4 min
# coop:     ~97,926 lines, 82 hero kills, 4,396 gold events, ~60.4 min
print('\n✅ Verification complete.')

In [ ]:
# ============================================================
# CELL 8 — Download outputs
# Download A: all raw TI match JSONs as a zip
# Download B: all combatlog txt files as a zip
# ============================================================
print('Packaging outputs...')

# Combatlog files (needed for Step 2)
combatlog_zip = '/content/combatlog_files.zip'
with zipfile.ZipFile(combatlog_zip, 'w') as zf:
    for fname in os.listdir(COMBATLOG_DIR):
        if fname.endswith('.txt'):
            zf.write(f'{COMBATLOG_DIR}/{fname}', fname)

# Raw TI JSONs (large — only download if you need to archive them)
n_json = len(os.listdir(RAW_DIR))
print(f'Raw JSON files in {RAW_DIR}: {n_json} files')
print('(Raw JSONs are large — Step 2 reads them directly from disk in Colab.')
print(' Only download them if you need a local archive.)')
print()

print('Downloading combatlog files (needed for Step 2)...')
files.download(combatlog_zip)

print('\n✅ Step 1 complete.')
print('Outputs for Step 2:')
print(f'  Combatlog txts: {combatlog_zip}  <- upload these in Step 2')
print(f'  Raw TI JSONs:   {RAW_DIR}/*.json  <- Step 2 reads these directly if run in same session')
print('\nIf running Step 2 in a new Colab session, re-run Cells 2-3 to re-download TI matches,')
print('then upload the combatlog zip from this step.')